# 10.01 · Error analysis first: read traces, build a failure taxonomy

**Module 10 · Testing, Evaluation & Observability** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 2–3 h |
| 📋 Prerequisites | Module 09 (the Orbita Cloud support desk) and Lesson 01.03 (`mastery.evals.stats`). Read the module README first. |
| 🔑 Keys | none (optional: `LANGSMITH_API_KEY` for the 🌐 cell) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-15 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Collect** a trace set that mixes realistic, edge-case and adversarial inputs.
2. **Open-code** traces: one free-form note per trace about the **first** thing that went wrong.
3. **Axial-code** the notes into 5–10 named failure categories, each with a definition, an example and a "not this".
4. **Rank** categories by frequency × severity, every frequency with a 95% interval, and know when to stop reading (saturation).
5. **Decide** per category: fix it now, catch it with a code check, or build a judge (Lesson 10.04).

## 📚 Official reading (verified 2026-10-15)

| Page | Read it for |
|---|---|
| [Evaluation concepts](https://docs.langchain.com/langsmith/evaluation-concepts) | Offline vs online evaluation, datasets, evaluators: the vocabulary of this module |
| [Annotate traces inline](https://docs.langchain.com/langsmith/annotate-traces-inline) | Adding notes and feedback to a trace while you read it |
| [Annotation queues](https://docs.langchain.com/langsmith/annotation-queues) | Reviewing many runs in a queue, with a rubric, as a team |
| [Query traces with the SDK](https://docs.langchain.com/langsmith/export-traces) | `Client.list_runs` (filters such as `is_root=True`) to pull traces for review |
| [Insights](https://docs.langchain.com/langsmith/insights) | Automatic clustering of traces (Plus and Enterprise plans only) |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "langsmith>=0.14.7,<1"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langgraph", "langsmith",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/10-testing-evaluation-observability/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm10-error-analysis'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m10-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 09 LIVE FAQ index): no usage telemetry.
# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# fastembed (Project 09 LIVE FAQ index) imports tqdm.auto; without ipywidgets it warns once. Progress bars still work.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

---
## Part 1 · Look before you measure

The tempting first eval is a score: "rate helpfulness 1–5" over every answer. It produces a number, and the number moves, but it can't tell you **what** to fix: a 3.7 is a mix of a dozen different problems, most of which you've never seen.

Error analysis turns it around. You read traces first, name the ways *this* system fails, count them, and only then choose evaluators, one per failure mode that matters. The method comes from qualitative research:

| Step | What you do | Output |
|---|---|---|
| **Collect** | Run the system on realistic, edge-case and adversarial inputs | 50–100 traces |
| **Open coding** | Read each trace; write one note about the **first** thing that went wrong | A label sheet with free-text notes |
| **Axial coding** | Group the notes into named categories with definitions | A failure taxonomy (5–10 categories) |
| **Count & rank** | Frequency × severity, with intervals | A priority list |
| **Decide** | Fix now, code check, or judge | The plan for Lessons 10.03–10.08 |

We use the support desk you built in Project 09 (handoffs pattern) on its offline model, so everyone reads the same traces. On your own system the steps are identical; only the traces change.

---
## Part 2 · Collect a trace set

`generate_inputs(60)` fills message templates with real customers and invoices: about half are **realistic** (what most customers write), a third **edge cases** (typos, two requests in one message, a missing amount, another language) and the rest **adversarial** (prompt injection, someone else's invoice). Edge and adversarial inputs are over-represented on purpose: that's where failures hide.

In [ ]:
# 🧪 Offline: the Orbita Cloud desk from Project 09 (handoffs pattern, rule-based offline model).
from langgraph.checkpoint.memory import InMemorySaver

from mastery.agents.support import SupportDesk, build_handoffs_desk, make_support_tools, offline_model
from mastery.agents.support_traces import generate_inputs, reference_label, reference_note, run_traces

OUT = REPO_ROOT / "data" / "outputs" / "m10"
OUT.mkdir(parents=True, exist_ok=True)
desk = SupportDesk.load()
app = build_handoffs_desk(offline_model, make_support_tools(desk), checkpointer=InMemorySaver())
inputs = generate_inputs(60, seed=10)
traces = run_traces(app, inputs, desk)
print(len(traces), "traces;", {k: sum(t["kind"] == k for t in traces) for k in ("realistic", "edge", "adversarial")})

**You should now see** `60 traces; {'realistic': 34, 'edge': 14, 'adversarial': 12}` (the mix is fixed by the seed).

In [ ]:
# 🧪 Offline: what a reviewer reads in a trace: the input, the tools in order, the refunds, the answer.
def show(trace):
    print(f"{trace['id']} [{trace['kind']}] {trace['input']}")
    print(f"   tools  : {trace['tools'] or '-'}")
    print(f"   refunds: {[(r['invoice_id'], r['amount']) for r in trace['refunds']] or '-'}")
    print(f"   answer : {trace['answer'][:110]}")

for trace in traces[:4]:
    show(trace)

**You should now see** four traces. Each record keeps only what you need to judge it: never the internal `want` field, which the generator uses to write the course's reference labels later.

### What just happened?

You have a **trace set**: real runs of the real system, frozen in a list. Reading raw LangSmith traces is the same activity with more detail (every model call, every tool call, tokens); Part 8 shows how to pull them with the SDK.

---
## Part 3 · Open coding: one note per trace, about the first failure

Read a trace and write **one** short note in your own words. Two rules make the notes usable later:

- **Write about the first thing that went wrong**, not every symptom. A missed customer ID causes a wrong answer, a useless follow-up and an annoyed customer; only the missed ID is the cause.
- **Don't invent categories yet.** "Asked for the ID again although the message had `c-1003`" is a good note; "NLU failure" is a category name in disguise.

Below are the first 12 traces. Three notes are filled in as examples. **Write the other nine** (leave `""` for a trace that went fine), then run the cell.

In [ ]:
# 🧪 Offline: YOUR open-coding notes for the first 12 traces ("" = no failure).
for trace in traces[:12]:
    show(trace)

MY_NOTES = {
    "T001": reference_note(traces[0], reference_label(traces[0])),   # example
    "T002": reference_note(traces[1], reference_label(traces[1])),   # example
    "T003": reference_note(traces[2], reference_label(traces[2])),   # example
    # "T004": "...",  write yours for T004–T012
}
print(f"\n{len(MY_NOTES)} of 12 notes written")

**You should now see** twelve traces and `3 of 12 notes written` (more once you add yours).

Reading 60 traces takes about an hour the first time. To keep the lesson moving, the next cell loads the **course's reference notes** for all 60 traces (written by reading each trace, like yours) and keeps any note you wrote. Compare yours with the reference afterwards: disagreements are exactly the definitions you'll need to sharpen.

In [ ]:
# 🧪 Offline: the label sheet: your notes where you wrote one, the reference notes elsewhere.
from mastery.evals import LabelSheet

rows = []
for trace in traces:
    label = reference_label(trace)
    rows.append({"trace_id": trace["id"], "note": MY_NOTES.get(trace["id"], reference_note(trace, label)),
                 "category": "", "kind": trace["kind"], "input": trace["input"],
                 "tools": " > ".join(trace["tools"]), "answer": trace["answer"][:200]})
sheet = LabelSheet(rows)
path = sheet.to_csv(OUT / "label-sheet.csv")
print("saved", path.relative_to(REPO_ROOT), "·", len(sheet), "rows ·", len(sheet.unlabelled()), "without a category yet")
for row in sheet.rows[:8]:
    print(f"  {row['trace_id']}: {row['note'] or '(no failure)'}")

**You should now see** `saved data/outputs/m10/label-sheet.csv · 60 rows · 60 without a category yet`, then eight notes. Open the CSV in a spreadsheet: the label sheet is **data**, and in Lesson 10.04 it becomes the set you check your judges against.

---
## Part 4 · Axial coding: from notes to a taxonomy

Now group the notes. Sort them, put similar ones together, and name each group. Each category needs four things, or it can't be measured consistently:

- a **definition** precise enough that two reviewers agree,
- a **real example** from your traces,
- a **"not this"** counter-example that sits close to the line,
- a **severity** from 1 (cosmetic) to 5 (harms a customer or the business).

Expect your definitions to change while you label (**criteria drift**): that's normal. When a definition changes, re-label the traces you already did.

Grouping the reference notes gives five categories. Compare them with the groups you'd have made:

In [ ]:
# 🧪 Offline: the reference taxonomy, with a decision per category (Part 7 explains them).
from mastery.agents.support_traces import REFERENCE_CATEGORIES
from mastery.evals import Taxonomy

ACTION = {"unsafe_action": "code_check", "asks_for_known_id": "fix", "stopped_at_lookup": "code_check",
          "dropped_request": "judge", "generic_fallback": "fix"}
taxonomy = Taxonomy.from_dicts({name: {**spec, "action": ACTION[name]} for name, spec in REFERENCE_CATEGORIES.items()})
for name, cat in taxonomy.categories.items():
    print(f"{name} (severity {cat.severity}): {cat.definition}")
    print(f"   not this: {cat.not_this}")
print("problems:", taxonomy.validate() or "none")

**You should now see** five categories with definitions and "not this" lines, and `problems: none`.

Now label every trace with one category (or `pass`). The reference labels follow the same rule as the notes: the first failure wins.

In [ ]:
# 🧪 Offline: axial coding = one category per row. Then check the sheet against the taxonomy.
for row, trace in zip(sheet.rows, traces):
    row["category"] = reference_label(trace)
sheet.to_csv(OUT / "label-sheet.csv")
print("problems:", taxonomy.validate(sheet) or "none")
print(sheet.counts().most_common())

**You should now see** `problems: none` and the counts: `pass` 46, then `asks_for_known_id` 7, `stopped_at_lookup` 4, `dropped_request` 2 and `generic_fallback` 1. `unsafe_action` doesn't appear: no trace in this sample moved money it shouldn't have.

### What just happened?

The notes became **data you can count**. `validate()` would have complained about a label that isn't in the taxonomy (a typo, or a category you forgot to define), an empty definition, or a trace left unlabelled.

---
## Part 5 · Count and rank: frequency × severity, with intervals

Which category matters most? Frequency alone favours harmless noise; severity alone favours rare disasters. Multiply them, and keep the uncertainty visible: with 60 traces, "7 of 60" means anything from about 6% to 22%.

In [ ]:
# 🧪 Offline: the priority list. Every share is a 95% Wilson interval (Lesson 01.03).
ranked = taxonomy.rank(sheet)
table([{"category": r.name, "count": r.count, "share (95% CI)": f"{r.share:.0%}".split(" (")[0],
        "severity": r.severity, "priority": f"{r.priority:.2f}", "action": r.action} for r in ranked],
      ["category", "count", "share (95% CI)", "severity", "priority", "action"])
print("\nany failure:", f"{sheet.failure_rate():.0%}")
_ = (OUT / "taxonomy.md").write_text(taxonomy.to_markdown(sheet), encoding="utf-8")

**You should now see** `asks_for_known_id` first (12% × 2 = 0.23) just ahead of `stopped_at_lookup` (7% × 3 = 0.20), then `dropped_request`, `generic_fallback`, and `unsafe_action` last with `0% [0%, 6%]`. Overall, about one trace in four fails: `23% [14%, 35%] (wilson, n=60)`. The top two are within each other's intervals: treat them as a tie and fix both. `taxonomy.md` is saved next to the label sheet.

### What just happened?

Look at the last row. **Zero observed is not zero risk:** with 0 failures in 60 traces, the interval still allows up to about 6% of traffic. That's why a severe category keeps a test even when you never saw it: Lesson 10.08 turns the Module 04–05 attacks into a red-team suite that runs on every pull request.

> ⚠️ **Production note: severity is a business decision, and "0 seen" is an interval.**
> Severities here are the course author's. In production, agree them with whoever owns the risk (support lead, finance, security), write them down next to the taxonomy, and report rare-but-severe categories by their **upper** bound ("at most 6% of traffic, 95% confidence"), not as "never happens".

---
## Part 6 · When to stop reading: saturation

You stop when new traces stop teaching you new failure modes. Track how many distinct categories you've seen after each trace, and read a fresh batch to test it.

In [ ]:
# 🧪 Offline: distinct categories after every 10 traces, then 60 NEW traces (ids T061-T120).
curve = dict(sheet.saturation())
print("first 60:", {n: curve[n] for n in (10, 20, 30, 40, 50, 60)})

more = run_traces(app, generate_inputs(60, seed=10, start=61), desk)
more_sheet = LabelSheet([{"trace_id": t["id"], "category": reference_label(t)} for t in more])
new = set(more_sheet.categories()) - set(sheet.categories())
print("new batch:", more_sheet.counts().most_common())
print("new categories in the second batch:", sorted(new) or "none")

**You should now see** `{10: 1, 20: 3, ..., 50: 3, 60: 4}`: three categories by trace 20, then a fourth (`generic_fallback`) only at trace 60. A category that shows up at the very end is a reason to read more, so the cell reads 60 more: `new categories in the second batch: none`. The counts move (`asks_for_known_id` 14 this time: sampling noise), the list of categories doesn't.

### What just happened?

A batch that adds no category is **saturation**: more reading would confirm what you know, not change it. Stop there for now, and re-run error analysis whenever the system or its traffic changes (a new prompt, a new model, a new kind of customer). Project 10 asks for at least 100 traces.

---
## Part 7 · Fix it or measure it

Not every category needs an LLM judge. For each one, ask in order:

1. **Is it a plain bug?** Then fix it now and add a regression test. `asks_for_known_id` is a case-sensitive regex (`C-\d{4}` misses `c-1003`); `generic_fallback` is a missing "I can't do that here, I'll pass you to a person" branch.
2. **Can code see it?** Then write a **code check**: cheap, exact, free to run on every trace. `stopped_at_lookup` is visible in the tool list ("refund" requested, `lookup_invoices` ran, `issue_refund` didn't); `unsafe_action` is visible in the refunds actually issued.
3. **Only then, a judge.** `dropped_request` needs to understand that a message contains two requests: language, not structure. That's a binary LLM judge (Lesson 10.04), checked against your labels before you trust it.

In [ ]:
# 🧪 Offline: a code check for `stopped_at_lookup`, compared with your labels. Exact and free.
def stopped_at_lookup(trace) -> bool:
    asked = "refund" in trace["input"].lower()
    return asked and "lookup_invoices" in trace["tools"] and "issue_refund" not in trace["tools"]

hits = [t["id"] for t in traces if stopped_at_lookup(t)]
labelled = [r["trace_id"] for r in sheet.rows if r["category"] == "stopped_at_lookup"]
print("code check flags:", len(hits), "· labelled:", len(labelled), "· agree:", hits == labelled)

**You should now see** `code check flags: 4 · labelled: 4 · agree: True`. When code can see a failure this clearly, a judge would only add cost and noise.

---
## Part 8 · 🌐 The same workflow on LangSmith traces

On a deployed system the traces live in LangSmith. Three tools from the reading list:

- **`Client.list_runs`** pulls root runs (one per conversation turn) to review in a notebook like this one.
- **Annotation queues** let you (or a team) review runs one by one with a rubric; their feedback scores are exportable like your label sheet.
- **Insights** clusters traces automatically into topics and failure patterns. It's a good second opinion on your taxonomy, but it needs a **Plus or Enterprise** plan (checked 2026-10-15), analyses at most **1,000** traces per report, and its model calls are billed to you (the docs estimate about $1–2 per 1,000 threads with OpenAI models).

> 🌐 Talks to LangSmith (free Developer plan is enough; no model tokens). Without `LANGSMITH_API_KEY` it prints `⏭️ Skipped`.

In [ ]:
# 🌐 LangSmith: the 20 most recent root runs of this lesson's project, as label-sheet rows.
if HAVE_LANGSMITH_KEY:
    from langsmith import Client

    runs = list(Client().list_runs(project_name=os.environ["LANGSMITH_PROJECT"], is_root=True, limit=20))
    live_rows = [{"trace_id": str(r.id), "note": "", "category": "", "input": str(r.inputs)[:200],
                  "answer": str(r.outputs)[:200]} for r in runs]
    print(len(live_rows), "runs ready to label (0 is fine if nothing was traced to this project yet)")
else:
    print("⏭️ Skipped: LANGSMITH_API_KEY is not set (see the setup cell).")

**You should now see** `⏭️ Skipped` without a key. With one, a count of runs (often `0` here, because this lesson's offline desk doesn't send traces; point `project_name` at a project with real traffic).

---
### 🧰 Jupyter skill: strip outputs and run notebooks headless

Two habits keep eval notebooks honest in git and CI:

- **`nbstripout`** removes outputs (and execution counts) when you commit, so diffs show code, not results, and no trace with customer data lands in git: `pip install nbstripout`, then `nbstripout --install` once in the repo (it adds a git filter).
- **Headless execution** runs a notebook top to bottom without a browser, as a smoke test: `jupyter nbconvert --to notebook --execute --output /tmp/run.ipynb modules/10-testing-evaluation-observability/10-01-error-analysis.ipynb`. The course's CI does the same for every notebook with no keys set.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| 20 categories, each with 1–2 traces | Categories named during open coding, or every symptom labelled | Note the **first** failure only; merge categories whose definitions overlap; aim for 5–10 |
| Two reviewers label the same trace differently | Definitions too vague, no "not this" | Add a counter-example at the line; re-label both sets; measure agreement |
| Category shares jump between batches | 60 traces give ±10-point intervals | Report intervals; read more traces for decisions that matter |
| Taxonomy only shows easy failures | Only realistic inputs were collected | Add edge-case and adversarial inputs on purpose (Part 2) |
| `taxonomy.validate()` lists unknown labels | A typo or a new category not added to the taxonomy | Fix the label or add the category with all four fields |
| Customer data ends up in git | Notebook outputs or label sheets committed | `nbstripout`; keep sheets under the git-ignored `data/outputs/`; mask PII (Lesson 10.02) |

## ⚠️ Production notes

> ⚠️ **Production note: review real traffic, and protect it.**
> The course uses generated inputs. In production, sample real traces (stratified by product area and by outcome), review them in annotation queues, and treat label sheets as customer data: access control, retention limits, PII masked before review (Lesson 10.02).

> ⚠️ **Production note: severity is a business decision, and "0 seen" is an interval.** (Part 5)

## 🏋️ Exercises

1. **(Easy)** Compare your nine notes from Part 3 with the reference labels (`reference_label(traces[i])`). Where you disagree, whose definition is clearer?
2. **(Medium)** Split `generic_fallback` into two categories, "out of scope" and "other language". Re-label, re-rank. Did the priority order change, and is either new category big enough to measure?
3. **(Medium)** Write a code check for `asks_for_known_id` and compare it with the labels, as Part 7 did for `stopped_at_lookup`.
4. **(Stretch)** Fix the bug behind `asks_for_known_id` in a copy of the desk: make the customer-ID regex in `mastery.agents.support` case-insensitive and normalize to upper case. Re-run the 60 traces. How many failures are left, with what interval?

<details>
<summary><b>Show solutions</b></summary>

**1.** Typical disagreements: a trace that both stopped at a lookup *and* ignored a second request (the first failure wins: the lookup); and passes that "felt" slow or wordy (not a failure in this taxonomy: write it down as a candidate category only if it keeps coming up).

**2.**
```python
def split(trace, label):
    if label != "generic_fallback":
        return label
    return "other_language" if "hola" in trace["input"].lower() else "out_of_scope"
sheet2 = LabelSheet([{"trace_id": t["id"], "category": split(t, reference_label(t))} for t in traces])
print(sheet2.counts())
```
With a single `generic_fallback` trace, each new category has 0–1 traces: far too few to measure. Keep one category, or read more traces first.

**3.**
```python
import re
def asks_for_known_id(trace):
    has_id = re.search(r"\bc-\d{4}\b", trace["input"], re.IGNORECASE)
    return bool(has_id) and "customer id" in trace["answer"].lower() and not trace["tools"]
print(sum(map(asks_for_known_id, traces)))   # 7, the same traces as the labels
```

**4.** After the fix the 7 `asks_for_known_id` traces become refunds or unlocks (or move to `stopped_at_lookup` when the message has no amount, a good example of "the first failure hides the next one"). Report the new failure rate with `wilson_interval`, and compare old and new on the **same** traces with `paired_compare` (Lesson 10.03).
</details>

## 🔎 Docs drill (5 min)

Which LangSmith plans include **Insights**, how many traces can one report analyse at most, and who pays for the model calls it makes?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Insights](https://docs.langchain.com/langsmith/insights): the first note says Plus and Enterprise plans.
2. The configuration section lists **Sample size** with a 1,000-trace limit.
3. The cost note says the model calls run on your own model provider key, with an estimate per 1,000 threads.

</details>

## ✅ Checkpoint

- [ ] I collected traces from realistic, edge-case and adversarial inputs.
- [ ] I wrote open-coding notes about the first failure of each trace I read.
- [ ] My taxonomy has 5–10 categories, each with a definition, an example, a "not this" and a severity.
- [ ] I ranked categories by frequency × severity and reported each share with an interval.
- [ ] I decided per category: fix, code check, or judge.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Check [Insights](https://docs.langchain.com/langsmith/insights) and [Annotation queues](https://docs.langchain.com/langsmith/annotation-queues) and log in [`docs/research-log.md`](../../docs/research-log.md): which plans include Insights (2026-10-15: Plus and Enterprise), its sample cap (1,000 traces), and whether annotation queues gained new rubric or reviewer features.

## Summary

- Look before you measure: generic scores hide *which* failure to fix.
- Open coding (one note, first failure) → axial coding (5–10 defined categories) → counts with intervals → frequency × severity.
- Saturation tells you when to stop reading; re-run error analysis when the system or traffic changes.
- Fix bugs, code-check what code can see, and keep judges for what needs language.
- The label sheet is data: it validates your judges in Lesson 10.04.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 10-02-observability.ipynb** → [`10-02-observability.ipynb`](10-02-observability.ipynb).

[Module 10 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)